### Prometheus and Grafana

**Prometheus** is an **open-source time-series database and monitoring system** designed for collecting and storing metrics.

#### Prometheus Key Characteristics:
- **Pull-based model**: Prometheus actively scrapes (pulls) metrics from configured endpoints
- **Time-series database**: Stores metrics as time-stamped values
- **Query language (PromQL)**: Powerful query language for analyzing metrics
- **Alerting**: Built-in alerting based on metric thresholds
- **Service discovery**: Automatically discovers services to monitor

#### Prometheus fetaures
✅ **Real-time monitoring**: See cache performance as code runs <br>
✅ **Historical data**: Compare different test runs over time <br>
✅ **Powerful queries**: Calculate hit rates, percentiles, speedups <br>
✅ **Industry standard**: Used by SAP, Oracle, SoundCloud, etc. <br>

**Grafana** is an **open-source data visualization and dashboarding platform** that creates interactive dashboards from metrics.

#### Grafana Key Characteristics:
- **Multi-datasource**: Connects to Prometheus, MySQL, PostgreSQL, Elasticsearch, etc.
- **Rich visualizations**: Time series, bar charts, pie charts, heatmaps, gauges, stats
- **Interactive dashboards**: Pan, zoom, drill-down, filter in real-time
- **Alerting**: Visual and notification-based alerts
- **Templating**: Dynamic dashboards with variables

#### Dashboard Features:
- **Panels**: Individual visualizations (graphs, charts, stats)
- **Rows**: Organize panels into collapsible sections
- **Variables**: Filter entire dashboard dynamically ($scenario, $pattern)
- **Annotations**: Mark important events on graphs
- **Time controls**: Select time ranges, refresh rates
- **Sharing**: Export to PDF, create snapshots, share links

#### Grafana features
✅ **Professional visualization**: Alternative for matplotlib for monitoring <br>
✅ **Real-time updates**: See metrics change as simulation runs <br>
✅ **Interactive exploration**: Zoom, filter, drill-down without code <br>
✅ **Industry standard**: Used by Uber, Electronic Arts, CERN etc. <br>


#### Workflow for Redis Cahce Demo

The application on 8080 (Python app) exposes metrics at /metrics endpoint
<br>📝 Example of exposed endpoints on localhost:8080: cache_hits_total, cache_latency_seconds.


Prometheus (port 9090) collects and stores metrics in time-series database
<br>📝 Prometheus Stores: metric_name{labels} value timestamp.

Grafana (port 3000) queries Prometheus for data and transforms metrics in visualizations in dashboards.
<br>📝 cache_hits_total{pattern="cache_aside",scenario="movie_recommendation"}
<br>📝 cache_read_latency_seconds_bucket{pattern="cache_aside",le="0.001"}
<br>📝 cache_read_latency_seconds_sum{pattern="cache_aside"}
<br>📝 cache_read_latency_seconds_count{pattern="cache_aside"}



### Setup

Run Redis, Prometheus and Grafana in a docker containers:

```
docker-compose up
```

To start a single Redis container:

```
docker run -d --name redis -p 6379:6379 redis:latest
```

#### Prometheus configuration
**prometheus.yml** configuration defines two scrape jobs that collect metrics from different sources:
- **redis job** for Redis internal metrics (memory usage, commands processed, key counts, etc.) is scraping the redis-exporter service on port 9121, which acts as a bridge that translates Redis INFO commands into Prometheus-compatible metrics;

- **demo-app job** collects custom application metrics (cache hits, misses, latency histograms) from your Jupyter notebook Python code running on the host machine at port 8000 via the prometheus_client library, using host.docker.internal as a special DNS name that allows the Prometheus container to reach services running on the host machine outside of Docker.

Every 5 seconds (the default scrape interval), Prometheus pulls metrics from both endpoints and stores them as time-series data that can be queried and visualized in Grafana.

#### Grafana configuration

```
volumes:
  - grafana-data:/var/lib/grafana
  - ./grafana/dashboards:/etc/grafana/provisioning/dashboards
  - ./grafana/datasources:/etc/grafana/provisioning/datasources
```

The Docker volume mounts enable Grafana's provisioning feature, which automatically configures datasources and dashboards on container startup without manual UI configuration.

In [ ]:
from prometheus_client import Counter, Histogram, Gauge, start_http_server
import threading, time

In [ ]:
# Prometheus Exporter
class PrometheusMetricsExporter:
    """Export cache metrics to Prometheus"""

    def __init__(self, port=8000):
        self.port = port
        self._server_started = False

        # Define metrics
        self.cache_hits = Counter('cache_hits_total', 'Total cache hits',
                                  ['pattern', 'scenario'])
        self.cache_misses = Counter('cache_misses_total', 'Total cache misses',
                                    ['pattern', 'scenario'])
        self.cache_evictions = Counter('cache_evictions_total', 'Total cache evictions',
                                       ['scenario'])

        self.cache_latency = Histogram('cache_read_latency_seconds', 'Cache read latency',
                                       ['pattern', 'scenario'])
        self.db_latency = Histogram('db_read_latency_seconds', 'Database read latency',
                                    ['pattern', 'scenario'])

        self.hit_rate = Gauge('cache_hit_rate_percent', 'Current cache hit rate',
                              ['pattern', 'scenario'])
        self.memory_usage = Gauge('cache_memory_usage_bytes', 'Cache memory usage',
                                  ['scenario'])

    def start(self):
        """Start Prometheus HTTP server in background thread"""
        if not self._server_started:
            def run_server():
                start_http_server(self.port)

            thread = threading.Thread(target=run_server, daemon=True)
            thread.start()
            self._server_started = True
            print(f"✓ Prometheus metrics server started on http://localhost:{self.port}")
            time.sleep(1)  # Give server time to start

    def record_hit(self, pattern: str, scenario: str, latency_seconds: float):
        self.cache_hits.labels(pattern=pattern, scenario=scenario).inc()
        self.cache_latency.labels(pattern=pattern, scenario=scenario).observe(latency_seconds)

    def record_miss(self, pattern: str, scenario: str, cache_latency_seconds: float,
                    db_latency_seconds: float):
        self.cache_misses.labels(pattern=pattern, scenario=scenario).inc()
        self.cache_latency.labels(pattern=pattern, scenario=scenario).observe(cache_latency_seconds)
        self.db_latency.labels(pattern=pattern, scenario=scenario).observe(db_latency_seconds)

    def record_eviction(self, scenario: str):
        self.cache_evictions.labels(scenario=scenario).inc()

    def update_hit_rate(self, pattern: str, scenario: str, hit_rate: float):
        self.hit_rate.labels(pattern=pattern, scenario=scenario).set(hit_rate)

    def update_memory_usage(self, scenario: str, bytes_used: int):
        self.memory_usage.labels(scenario=scenario).set(bytes_used)


print("✓ Prometheus exporter class defined")

In [ ]:
prom_exporter = PrometheusMetricsExporter(port=8000)
prom_exporter.start()

In [ ]:
prom_exporter.record_hit(pattern = "cache_aside", scenario = "movie_recommendation", latency_seconds = 1)
prom_exporter.record_hit(pattern = "cache_aside", scenario = "movie_recommendation", latency_seconds = 2)
prom_exporter.record_miss("cache_aside", "movie_recommendation", cache_latency_seconds = 1, db_latency_seconds  = 3)
prom_exporter.record_miss("cache_aside", "movie_recommendation", cache_latency_seconds = 1, db_latency_seconds  = 4)

### Metric Types in Prometheus

#### 1. **Counter** (Always Increasing)

```python
cache_hits_total = Counter('cache_hits_total', 'Total cache hits')
cache_hits_total.inc()  # Increment by 1
```

- Use for: Total hits, total requests, total errors
- Example: `cache_hits_total` 0 → 100 → 250 → 500 ...

#### 2. **Gauge** (Measurment, Can Go Up or Down)
```python
cache_hit_rate = Gauge('cache_hit_rate_percent', 'Current hit rate')
cache_hit_rate.set(75.5)  # Set to specific value
```
- Use for: Current values, percentages, temperatures
- Example: `cache_hit_rate_percent` 50 → 75 → 60 → 85


#### 3. **Histogram** (Distribution of Values)

```python
cache_latency = Histogram('cache_read_latency_seconds', 'Cache read latency')
cache_latency.observe(0.0023)  # Record observation
```
- Use for: Response times, request sizes
- Automatically creates: sum, count, buckets (for percentiles)
- Example: Calculate P50, P95, P99 latencies

### PromQL: Prometheus Query Language

#### Basic Queries:
```promql
# Get current value
cache_hit_rate_percent

# Get rate per second using a frame of 1 minute
rate(cache_hits_total[1m])

# Calculate hit rate
rate(cache_hits_total[1m]) /
(rate(cache_hits_total[1m]) + rate(cache_misses_total[1m])) * 100

# Get 95th percentile latency
histogram_quantile(0.95, rate(cache_read_latency_seconds_bucket[5m]))

# Aggregate across labels
sum by(pattern) (rate(cache_hits_total[1m]))

# Filter by label
cache_hits_total{scenario="heavy_repeat"}
```